In [1]:
import pandas as pd
import scipy.stats as stats

pd.set_option('display.max_columns', None)
import numpy as np

from sklearn.ensemble import RandomForestRegressor
import xgboost
from pyprojroot import here

import seaborn as sns
sns.set_style("whitegrid")

from data_utils import sort_data, colors_config
from model_utils import ModelSelector, train_test_split, one_hot_encode, set_seed, FPLDataPipe
import config


In [2]:
set_seed(77)
data = pd.read_parquet(config.FPL_DATA_PATH)

In [3]:
data.head()

,position_id,second_name_id,status_id,first_name_id,match_id,team_name_id,team_code_id,gw_id,season_id,web_name_id,player_code_id,team_strength,position,was_home,cost_change_event_lagged_1,transfers_in_event_lagged_1,accurate_crosses_lagged_1,accurate_long_balls_ema_8,transfers_out_event_ema_8,xg_lagged_1,xgot_lagged_1,assists_lagged_1,saves_ema_8,high_claim_ema_8,fouls_committed_ema_8,form_lagged_1,xg_ema_8,event_points,was_fouled_lagged_1,chances_created_lagged_1,gk_accurate_long_balls_lagged_1,successful_dribbles_ema_8,goals_ema_8,touches_opposition_box_ema_8,start_min_ema_8,gk_accurate_long_balls_ema_8,ground_duels_won_ema_8,interceptions_ema_8,shots_on_target_ema_8,recoveries_lagged_1,value_season_lagged_1,xa_ema_8,big_chances_missed_ema_8,sweeper_actions_ema_8,touches_lagged_1,offsides_ema_8,accurate_crosses_percent_ema_8,minutes_played_lagged_1,xg_per_90_ema_8,final_third_passes_ema_8,shots_on_target_lagged_1,event_points_lagged_1,bonus_ema_8,xgot_faced_lagged_1,value_ratio_ema_8,ep_this_ema_8,sweeper_actions_lagged_1,high_claim_lagged_1,tackles_ema_8,event_points_ema_8,team_goals_conceded_lagged_1,value_form_lagged_1,penalties_scored_ema_8,transfers_in_event_ema_8,accurate_crosses_ema_8,bps_lagged_1,accurate_passes_lagged_1,gk_accurate_passes_ema_8,headed_clearances_ema_8,goals_conceded_ema_8,xa_per_90_ema_8,ep_this_lagged_1,dribbled_past_lagged_1,aerial_duels_won_ema_8,dreamteam_count_lagged_1,duels_won_ema_8,goals_lagged_1,final_third_passes_lagged_1,dribbled_past_ema_8,expected_goal_involvements_per_90_ema_8,xgot_ema_8,team_goals_conceded_ema_8,was_fouled_ema_8,ep_next_lagged_1,duels_won_lagged_1,form_ema_8,cost_change_start_lagged_1,recoveries_ema_8,big_chances_missed_lagged_1,saves_lagged_1,now_cost_ema_8,ict_index_ema_8,minutes_played_ema_8,transfers_in_ema_8,goals_conceded_lagged_1,touches_ema_8,clearances_lagged_1,total_shots_ema_8,selected_rank_ema_8,value_form_ema_8,assists_ema_8,tackles_lagged_1,aerial_duels_won_lagged_1,penalties_missed_ema_8,duels_lost_ema_8,transfers_out_ema_8,transfers_out_event_lagged_1,now_cost_rank_lagged_1,clearances_ema_8,ground_duels_won_lagged_1,expected_assists_lagged_1,expected_goals_conceded_ema_8,start_min_lagged_1,touches_opposition_box_lagged_1,penalties_order_lagged_1,selected_by_percent_ema_8,headed_clearances_lagged_1,duels_lost_lagged_1,total_shots_lagged_1,accurate_passes_ema_8,xa_lagged_1,accurate_long_balls_lagged_1,successful_dribbles_lagged_1,chances_created_ema_8,fouls_committed_lagged_1,interceptions_lagged_1,threat_lagged_1,influence_lagged_1
0,Midfielder,Milner,a,James,24-25-prem-everton-vs-brighton-&-hove-albion,Brighton,36,1,2425,Milner,15157,3,MID,0,0,0,0,0.000000,0.000000,0.0,0.0,0,0.0,0.0,0.0,0.0,0.000000,2,0,0,0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0,0.0,0.000000,0.0,0.0,0,0.0,0.000000,0,0.000000,0.000000,0,0,0.0,0.0,0.000000,0.000000,0,0,0.000000,0.000000,0,0.0,0.0,0.000000,0.000000,0,0,0.0,0.0,0.0,0.000000,0.0,0,0.0,0,0.000000,0,0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0,0.000000,0,0.000000,0,0,0.0,0.000000,0.0,0.000000,0,0.000000,0,0.000000,0.000000,0.000000,0.0,0,0,0.0,0.000000,0.000000,0,0,0.000000,0,0.0,0.000000,0,0,0,0.0,0,0,0,0.000000,0.0,0,0,0.000000,0,0,0,0.0
1,Midfielder,Milner,a,James,24-25-prem-brighton-&-hove-albion-vs-mancheste...,Brighton,36,2,2425,Milner,15157,3,MID,1,0,513,2,1.000000,421.000000,0.0,0.0,0,0.0,0.0,1.0,2.0,0.000000,2,0,3,0,0.000000,0.0,2.000000,0.0,0.0,4.000000,0.000000,0.000000,0,0.4,0.110000,0.0,0.0,48,0.0,29.000000,82,0.000000,2.000000,0,2,0.0,0.0,0.400000,1.600000,0,0,4.000000,2.000000,0,0.4,0.0,513.000000,2.000000,10,28,0.0,0.0,0.0,0.120732,1.6,0,0.0,0,4.000000,0,2,0.0,0.120732,0.0,0.000000,0.000000,1.6,4,2.000000,0,0.000000,0,0,5.0,7.000000,82.0,513.000000,0,48.000000,0,0.000000,471.000000,0.400000,0.0,4,0,0.0,2.000000,421.000000,421,172,0.000000,4,0.11,0.450000,0,2,6,0.1,0,2,0,28.000000,0.11,1,0,3.000000,1,0,4,22.4
2,Midfielder,Milner,d,James,24-25-prem-arsenal-vs-brighton-&-hove-albion,Brighton,36

In [4]:
data.shape

(20820, 128)

In [5]:
cols_map = colors_config.FEATURES_GROUP
cols_map.keys()

dict_keys(['target', 'cols_id', 'static_cols', 'order_cols', 'ratio_features', 'performance_cols', 'pre_game_cols'])

In [6]:
num_cols = [c for c in data.columns if "ema" in c or "lagg" in c]
len(num_cols)

113

In [7]:
static_cols = [f"{c}_id" for c in cols_map["static_cols"] if c not in  ["was_home", "team_strength"]] + ["was_home", "team_strength"]

In [8]:
fpl_data_pipe = FPLDataPipe(num_cols=num_cols,
                            cat_cols=static_cols)

data_splits = train_test_split(data)

X_train = fpl_data_pipe.preprocessor.fit_transform(data_splits["X_train"])
X_valid = fpl_data_pipe.preprocessor.transform(data_splits["X_valid"])
X_test = fpl_data_pipe.preprocessor.transform(data_splits["X_test"])

y_train, y_valid, y_test = data_splits["y_train"], data_splits["y_valid"], data_splits["y_test"]

In [9]:
X_train.shape, y_train.shape, X_valid.shape, y_valid.shape, X_test.shape, y_test.shape

((11567, 129), (11567,), (4465, 129), (4465,), (4788, 129), (4788,))

In [10]:
X_train.head()

,num__cost_change_event_lagged_1,num__transfers_in_event_lagged_1,num__accurate_crosses_lagged_1,num__accurate_long_balls_ema_8,num__transfers_out_event_ema_8,num__xg_lagged_1,num__xgot_lagged_1,num__assists_lagged_1,num__saves_ema_8,num__high_claim_ema_8,num__fouls_committed_ema_8,num__form_lagged_1,num__xg_ema_8,num__was_fouled_lagged_1,num__chances_created_lagged_1,num__gk_accurate_long_balls_lagged_1,num__successful_dribbles_ema_8,num__goals_ema_8,num__touches_opposition_box_ema_8,num__start_min_ema_8,num__gk_accurate_long_balls_ema_8,num__ground_duels_won_ema_8,num__interceptions_ema_8,num__shots_on_target_ema_8,num__recoveries_lagged_1,num__value_season_lagged_1,num__xa_ema_8,num__big_chances_missed_ema_8,num__sweeper_actions_ema_8,num__touches_lagged_1,num__offsides_ema_8,num__accurate_crosses_percent_ema_8,num__minutes_played_lagged_1,num__xg_per_90_ema_8,num__final_third_passes_ema_8,num__shots_on_target_lagged_1,num__event_points_lagged_1,num__bonus_ema_8,num__xgot_faced_lagged_1,num__value_ratio_ema_8,num__ep_this_ema_8,num__sweeper_actions_lagged_1,num__high_claim_lagged_1,num__tackles_ema_8,num__event_points_ema_8,num__team_goals_conceded_lagged_1,num__value_form_lagged_1,num__penalties_scored_ema_8,num__transfers_in_event_ema_8,num__accurate_crosses_ema_8,num__bps_lagged_1,num__accurate_passes_lagged_1,num__gk_accurate_passes_ema_8,num__headed_clearances_ema_8,num__goals_conceded_ema_8,num__xa_per_90_ema_8,num__ep_this_lagged_1,num__dribbled_past_lagged_1,num__aerial_duels_won_ema_8,num__dreamteam_count_lagged_1,num__duels_won_ema_8,num__goals_lagged_1,num__final_third_passes_lagged_1,num__dribbled_past_ema_8,num__expected_goal_involvements_per_90_ema_8,num__xgot_ema_8,num__team_goals_conceded_ema_8,num__was_fouled_ema_8,num__ep_next_lagged_1,num__duels_won_lagged_1,num__form_ema_8,num__cost_change_start_lagged_1,num__recoveries_ema_8,num__big_chances_missed_lagged_1,num__saves_lagged_1,num__now_cost_ema_8,num__ict_index_ema_8,num__minutes_played_ema_8,num__transfers_in_ema_8,num__goals_conceded_lagged_1,num__touches_ema_8,num__clearances_lagged_1,num__total_shots_ema_8,num__selected_rank_ema_8,num__value_form_ema_8,num__assists_ema_8,num__tackles_lagged_1,num__aerial_duels_won_lagged_1,num__penalties_missed_ema_8,num__duels_lost_ema_8,num__transfers_out_ema_8,num__transfers_out_event_lagged_1,num__now_cost_rank_lagged_1,num__clearances_ema_8,num__ground_duels_won_lagged_1,num__expected_assists_lagged_1,num__expected_goals_conceded_ema_8,num__start_min_lagged_1,num__touches_opposition_box_lagged_1,num__penalties_order_lagged_1,num__selected_by_percent_ema_8,num__headed_clearances_lagged_1,num__duels_lost_lagged_1,num__total_shots_lagged_1,num__accurate_passes_ema_8,num__xa_lagged_1,num__accurate_long_balls_lagged_1,num__successful_dribbles_lagged_1,num__chances_created_ema_8,num__fouls_committed_lagged_1,num__interceptions_lagged_1,num__threat_lagged_1,num__influence_lagged_1,col__status_id_a,col__status_id_d,col__status_id_i,col__status_id_n,col__status_id_s,col__status_id_u,col__position_id_Defender,col__position_id_Forward,col__position_id_Goalkeeper,col__position_id_Midfielder,col__was_home_0,col__was_home_1,col__team_strength_2.0,col__team_strength_3.0,col__team_strength_4.0,col__team_strength_5.0
0,0.5,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.119760,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.032051,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.147059,0.0,0.0,0.126214,0.068851,0.0,0.0,0.000000,0.090909,0.000000,0.153846,0.0,0.000000,0.000000,0.018683,0.000000,0.0,0.0,0.0,0.000000,0.064151,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.063830,0.000000,0.125000,0.380952,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.153846,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.00000,

In [11]:
xgb = xgboost.XGBRegressor(random_state=77, objective='reg:squarederror', eval_metric='rmse')
rf = RandomForestRegressor(random_state=77)

In [12]:
param_grid = [
    {
        "n_estimators": stats.randint(300, 800),
        "max_depth": stats.randint(4, 8),
        "min_samples_split": stats.randint(10, 40),
        "min_samples_leaf": stats.randint(2, 15)
    },
    {
        "n_estimators": stats.randint(300, 1000),
        "max_depth": stats.randint(2, 5),
        "learning_rate": stats.uniform(0.005, 0.05),
        "reg_alpha": stats.uniform(0.1, 10.0),
        "reg_lambda": stats.uniform(0.1, 10.0),
        "colsample_bytree": stats.uniform(0.5, 0.4),
        "subsample": stats.uniform(0.6, 0.3)
    }
]

In [13]:
model = [rf, xgb]
model_names = ["rf", "xgb"]


In [14]:
selector = ModelSelector(random_state=77)

In [15]:
results, best_params = selector.params_search(models=model,
                       models_names=model_names,
                       params_grid=param_grid,
                       X_train=X_train,
                       y_train=y_train,
                       cv=5,
                       n_iter=5,
                       scoring="neg_mean_squared_error")


Fitting 5 folds for each of 5 candidates, totalling 25 fits
Fitting 5 folds for each of 5 candidates, totalling 25 fits


In [16]:
results

,data,name,cv_mean_neg_mean_squared_error,best_params,mse,mae,r2
1,train,xgb,-7.667213,"{'colsample_bytree': 0.5255086096322625, 'lear...",7.397352,1.926383,0.156354
0,train,rf,-7.708117,"{'max_depth': 7, 'min_samples_leaf': 7, 'min_s...",6.778221,1.850329,0.226964


In [17]:
eval_results, y_preds = selector.evaluate(best_params, X_valid, y_valid)

In [18]:
eval_results

,data,model_name,mse,mae,r2
0,test,rf,8.594006,2.080455,0.051473
1,test,xgb,8.514025,2.066234,0.060300
